# 4b. EuroCrops: prepare carrot references and collect annual imagery

Use the local 2021 parcel declarations to collect annual Sentinel-2 imagery for development. CropHarvest remains the reserved test source. Run this notebook alongside 02–04, then rerun 05–07 after collection completes.


## 1. Setup and configuration

Review the local parcel path, batch size, and collection mode before running.

In [ ]:
# Review these settings before Run All.
import os
import sys
from pathlib import Path

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
sys.path.insert(0, str(root / "src"))
from crop_pipeline import notebook_setup

run_satellite_collection = True
submit_only = True  # Queue exports. Set False to collect finished results.
extra_batches = 14  # Total extra batch slots for this round.
parcels_per_batch = 100
parcel_path = root / "data/external/eurocrops_v2/NL_2021.parquet"
notebook_setup.prepare_environment(root, install_missing=run_satellite_collection)
os.chdir(root)
settings_path = root / "settings.local.json"
settings = notebook_setup.ensure_local_settings(settings_path)
report_root = (root / Path(settings.get("report_root", "sen4agrinet_work/inventory")).expanduser()).resolve()


Notebook environment ready: /Users/josedavid/Documents/programming/external-crop-data-classification/.venv/bin/python


## 2. Obtain references

Use the local Dutch EuroCrops parcel declarations for 2021.

In [ ]:
from crop_pipeline.sources import adaptive, annual, eurocrops
from crop_pipeline.windows.seasons import file_sha256

print("Local EuroCrops parcel file:", parcel_path)


Local EuroCrops parcel file: /Users/josedavid/Documents/programming/external-crop-data-classification/data/external/eurocrops_v2/NL_2021.parquet


## 3. Validate references

Select declared carrot production parcels with valid footprints.

In [ ]:
features = eurocrops.load_carrots(parcel_path)
print("Eligible declared carrot references:", len(features))


Eligible declared carrot references: 2194


## 4. Prepare the collection round

Resume saved plans and select unused carrot parcels for new batches.

In [ ]:
batches = adaptive.prepare_round_batches(
    eurocrops.SOURCE, features, annual.archive_root(settings, report_root), report_root,
    provenance={"dataset": "EuroCrops", "country": "NL", "year": eurocrops.YEAR,
                "parcel_file_sha256": file_sha256(parcel_path),
                "carrot_production_codes": sorted(eurocrops.CARROT_PRODUCTION_CODES)},
    parcels_per_crop=parcels_per_batch, candidate_crops=("carrot",), extra_batches=extra_batches,
)


Round 2: resuming saved eurocrops selections and original collection plans.
Round 2: up to 100 candidates per crop in each new batch. Resume/submit 3 batches.


## 5. Collect annual imagery

Use `submit_only = True` to queue exports. Use `False` to download finished results.

In [ ]:
from crop_pipeline import source_workflow

project = notebook_setup.project_for_collection(settings, settings_path, required=run_satellite_collection)
result = adaptive.collect_batches(
    batches, project, enabled=run_satellite_collection, authenticate=run_satellite_collection,
    submit_only=submit_only, download_workers=2, intervals_per_request=6,
)


eurocrops 2021: selecting tiles 1–132 of 132
Scene inventories: 25/132 (0.0s)
Scene inventories: 50/132 (0.0s)
Scene inventories: 75/132 (0.1s)
Scene inventories: 100/132 (0.1s)
Scene inventories: 125/132 (0.1s)
Scene inventories: 132/132 (0.1s)
  Resuming with 11,000 rows from 11 verified cached pages.
  Fetching page 12 (up to 5,000 rows; 11,000 saved so far)…
  Page 12 verified: 1,000 rows; 12,000 total (0.6s).
  Fetching page 13 (up to 5,000 rows; 12,000 saved so far)…
  Page 13 verified: 1,000 rows; 13,000 total (0.6s).
  Fetching page 14 (up to 5,000 rows; 13,000 saved so far)…
  Page 14 verified: 1,000 rows; 14,000 total (0.5s).
  Fetching page 15 (up to 5,000 rows; 14,000 saved so far)…
  Page 15 verified: 1,000 rows; 15,000 total (0.5s).
  Fetching page 16 (up to 5,000 rows; 15,000 saved so far)…
  Page 16 verified: 1,000 rows; 16,000 total (0.7s).
  Fetching page 17 (up to 5,000 rows; 16,000 saved so far)…
  Page 17 verified: 1,000 rows; 17,000 total (0.9s).
  Fetching page 1

## 6. Summary and next action

Check the batch status below. Once collection is complete, rerun Notebooks 05–07 to include EuroCrops in the development features.

In [ ]:
source_workflow.print_summary(eurocrops.SOURCE, batches, result, "reuse", 10)


eurocrops — completed
References: reuse | Inward buffer: 10 m
Selected parcels: carrot 300
Batches: eurocrops_extra_0004: completed; eurocrops_extra_0005: completed; eurocrops_overnight_100: completed
Verified local tiles: 398/398 (reported batch ranges)
Cloud jobs: 0 queued, 0 running, 0 ready to download, 0 failed, 0 cancelled, 0 unknown
Campaign: /Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/adaptive_training/829e0cec79b8
Annual TIFF root: /Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/annual_tifs
Saved registries: /Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/adaptive_training/829e0cec79b8/rounds/0002/eurocrops_extra_0004/source_handoffs/full_year_all_parcels/eurocrops.json, /Users/josedavid/Documents/programming/external-crop-data-classification/sen4agrinet_work/adaptive_training/829e0cec79b8/rounds/0002/eurocrops_extra_0005/source_handoffs/full_year_a